# Create Dave config

Builds two independent Dave recipe XMLs from `round_info.csv` (notebook 03)
and the Kilroy config for `MICROSCOPE`, in acquisition order: cells, hybs.
The focus-lock test recipe, which runs before them, is notebook 02c. Each
section's printed cost estimate assumes it starts right after the previous
one finishes -- see `CELLS_LEAD_MIN` below.

In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime, timedelta
import pandas as pd
from IPython.display import display

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent  # MERci/ (notebook lives in MERci/notebooks/before_imaging/regular/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.acquisition.dave   import (
    create_dave_config, dave_config_filename, dave_cells_config_filename,
    experiment_estimate_table, annotate_dave_with_round_info,
)
from MERci.acquisition.kilroy import find_kilroy_config
from MERci.acquisition.display import display_xml
from MERci.acquisition.pipeline_config import load_pipeline_config

# PIPELINE_ID names this variant's own pipeline.yaml -- see notebook 01's own
# comment on PIPELINE_ID/PIPELINE_CONFIG for the rationale.
PIPELINE_ID     = "tumor_epi"   # EDIT ME -- one of the ids in pipeline_export.PIPELINES
PIPELINE_CONFIG = load_pipeline_config(MERCI_DIR / "data" / "pipelines" / f"{PIPELINE_ID}_pipeline.yaml")

In [ ]:
SETTINGS_DIR  = SAMPLE_DIR / "settings"
METADATA_DIR  = SAMPLE_DIR / "metadata"
POSITIONS_DIR = SAMPLE_DIR / "positions"

# SAMPLE_NAME is the TRUE top-level experiment id -- must match what notebooks
# 02/03 used, since the positions files and round_info rows below are named
# with it (see notebook 02's docstring for why this isn't SAMPLE_DIR.name).
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
# POSITIONS_TAG is what every positions_{...}.txt filename this notebook reads/builds
# actually uses -- SAMPLE_NAME alone in the flat layout, or SAMPLE_NAME_IMAGING_DIR in
# a split layout.
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SAMPLE_NAME  : {SAMPLE_NAME}")

In [ ]:
# ── Experiment parameters ──────────────────────────────────────────
MICROSCOPE           = PIPELINE_CONFIG.microscope   # must match what you set in notebooks 01/03
USE_ADAPTORS         = PIPELINE_CONFIG.use_adaptors           # True = adaptor-based fluidics; False = direct readouts
INCLUDE_FINAL_CLEAVE = PIPELINE_CONFIG.include_final_cleave   # True = add a final cleave step after last imaging round
FIRST_HYB_NO_CLEAVE  = PIPELINE_CONFIG.first_hyb_no_cleave    # True = first hyb (after the cells round) omits the cleave step

# Default round structure: imaging round 1 = cells only; rounds 2..N_HYBS+1 = bits #1..#N.
# The fluidics before the first bits round has no cleave; later fluidics include the cleave.

# ── Read what notebook 03 produced ──────────────────────────────────
round_info_path = METADATA_DIR / "round_info.csv"
if not round_info_path.exists():
    raise FileNotFoundError(f"{round_info_path} not found -- run notebook 03 first.")
round_info = pd.read_csv(round_info_path)

rbc_path = METADATA_DIR / "round_bit_color_map.csv"
if not rbc_path.exists():
    raise FileNotFoundError(f"{rbc_path} not found -- run notebook 03 first.")
N_HYBS = int(pd.read_csv(rbc_path)["round"].max())

MULTI_BOUNDARY = "positions_file" in round_info.columns
print(f"N_HYBS               : {N_HYBS}")
print(f"Multi-boundary layout: {MULTI_BOUNDARY}")
print(f"Use adaptors         : {USE_ADAPTORS}")
print(f"Final cleave         : {INCLUDE_FINAL_CLEAVE}")
print(f"First hyb no cleave  : {FIRST_HYB_NO_CLEAVE}")

In [ ]:
# ── Resolve positions inputs for the recipe ──────────────────────────────
if MULTI_BOUNDARY:
    # Per-segment: each round_info row names its own positions file in POSITIONS_DIR.
    positions_arg     = None
    positions_dir_arg = POSITIONS_DIR
    missing = [f for f in round_info["positions_file"].unique()
               if not (POSITIONS_DIR / f).exists()]
    if missing:
        raise FileNotFoundError(
            f"Positions files referenced by round_info are missing (run notebook 02): {missing}"
        )
else:
    # Single-positions: one file for every movie.
    positions_arg     = POSITIONS_DIR / f"positions_{POSITIONS_TAG}.txt"
    positions_dir_arg = None
    if not positions_arg.exists():
        raise FileNotFoundError(f"Positions file not found: {positions_arg}")

# Resolve the Kilroy config that will run this experiment. Its protocol names are
# the source of truth for the fluidic steps written into the Dave recipe, so every
# protocol referenced is guaranteed to exist in Kilroy. If the microscope has no
# Kilroy config, fall back to MF2's.
KILROY_DIR    = MERCI_DIR / "data" / "configs" / "kilroy"
KILROY_CONFIG = find_kilroy_config(MICROSCOPE, KILROY_DIR, fallback_microscope="MF2")
print(f"Kilroy config (protocol source): {KILROY_CONFIG.name}")

# Cells is always imaging_round 1; every later round is a hyb/bits round --
# see this notebook's own "Experiment parameters" cell above.
cells_round_info = round_info[round_info["imaging_round"] == 1]
hybs_round_info  = round_info[round_info["imaging_round"] > 1]

## Cells

Writes the cells-only Dave recipe: a single imaging round, no fluidics at
all (the fluidics that hybridizes bit 1's probes belongs to the hybs recipe
below instead, as its own leading step). Independently runnable in Dave on
its own.

In [ ]:
# ── Build the cells recipe ────────────────────────────────────────────────
cells_output = SETTINGS_DIR / dave_cells_config_filename(MICROSCOPE, SAMPLE_NAME)

CELLS_LEAD_MIN = 30   # minutes from now until this recipe is assumed to start on
                       # the microscope -- cascades into the hybs start time below
cells_start = datetime.now() + timedelta(minutes=CELLS_LEAD_MIN)

cells_est = create_dave_config(
    round_info            = cells_round_info,
    positions_file        = positions_arg,
    settings_dir          = SETTINGS_DIR,
    output_path           = cells_output,
    kilroy_config         = KILROY_CONFIG,
    positions_dir         = positions_dir_arg,
    include_final_cleave  = False,   # no fluidics belongs in the cells-only file
    microscope            = MICROSCOPE,
    start_time            = cells_start,
    per_round             = False,   # per-round breakdown shown as a table at the end instead
)

print(f"Cells Dave config saved: {cells_output}")
display_xml(cells_output)

## Hybs

Writes the hybs-only Dave recipe: every bits/hyb round plus its between-round
fluidics, PLUS its own leading fluidics block before hyb 1
(`leading_fluidics=True`) -- so this file is self-contained and independently
runnable in Dave without depending on the cells recipe above having just
finished in the same session.

Each `Hyb NN Fluidics` loop is annotated with an XML comment naming the
bits/colors imaged in the round it precedes (from notebook 03's
`round_bit_color_map.csv`), so the recipe is self-documenting when read or
edited later. The cells recipe above images no bits, so it gets no such
comments.

In [ ]:
# ── Build the hybs recipe ─────────────────────────────────────────────────
dave_output = SETTINGS_DIR / dave_config_filename(MICROSCOPE, N_HYBS, SAMPLE_NAME)

# Assumed to start right after the cells recipe above finishes.
hybs_start = cells_start + timedelta(seconds=cells_est.total_time_s)

hybs_est = create_dave_config(
    round_info            = hybs_round_info,
    positions_file        = positions_arg,
    settings_dir          = SETTINGS_DIR,
    output_path           = dave_output,
    use_adaptors          = USE_ADAPTORS,
    include_final_cleave  = INCLUDE_FINAL_CLEAVE,
    first_hyb_no_cleave   = FIRST_HYB_NO_CLEAVE,
    leading_fluidics      = True,
    kilroy_config         = KILROY_CONFIG,
    positions_dir         = positions_dir_arg,
    microscope            = MICROSCOPE,
    start_time            = hybs_start,
    per_round             = False,   # per-round breakdown shown as a table at the end instead
)

print(f"Hybs Dave config saved: {dave_output}")

# round_bit_color_map.csv is hyb/bit-indexed (1..N), but this recipe images
# bits in imaging rounds 2..N+1 (round 1 = cells), so shift +1 to line up
# with the Hyb NN Fluidics loops (rbc_path resolved in "Experiment parameters").
rbc_df       = pd.read_csv(rbc_path)
annotate_rbc = [(int(r) + 1, int(b), int(c))
                for r, b, c in rbc_df[["round", "bit", "color"]].itertuples(index=False, name=None)]
annotate_dave_with_round_info(dave_output, annotate_rbc)
print(f"Annotated with {len(annotate_rbc)} bit/color comment(s)")
display_xml(dave_output)

## Combined timing table

Block-by-block start/end/duration across both recipes above (cells, hybs),
in run order -- same table style as
`during_imaging/dave_timing_accuracy.ipynb`'s own block-by-block table.

In [ ]:
timing_table = pd.concat([
    experiment_estimate_table(cells_est, cells_start),
    experiment_estimate_table(hybs_est, hybs_start),
], ignore_index=True)

display(timing_table)